<div align="center">

# 🎬 OmniUpscaler (OmniScale AI) Pro Max Studio (2026)
**Production-Grade AI Video Super-Resolution on Cloud GPUs — 100% Free & Open Source**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiL4gh/omni-upscaler/blob/main/enhanced_video_upscaler.ipynb)
[![GitHub License](https://img.shields.io/badge/license-MIT-blue.svg)](https://github.com/NiL4gh/omni-upscaler/blob/main/LICENSE)
[![Zero Hardware](https://img.shields.io/badge/Hardware-0%25%20Local%20GPU%20Required-success.svg)](https://github.com/NiL4gh/omni-upscaler)

Runs completely in the cloud on Google Colab (Free NVIDIA T4 GPU). **Works on low-end PCs, laptops, tablets, and mobile phones.**

</div>

---

### 🚀 Quickstart
1. Click **Connect** in the top right (ensure Runtime is set to **T4 GPU**).
2. Click the **Play button ▶** on the cell below to start the studio.
3. Click the generated **public link (`https://...gradio.live`)** to open the full UI on desktop or mobile.


In [ ]:
#@title 🎬 Launch OmniUpscaler Pro Studio { display-mode: "form" }
# ==============================================================================
# 🚀 OMNIUPSCALE PRO STUDIO 2026 (UI/UX Pro Max Edition)
# Glassmorphism, Micro-Interactions, Zero-Clutter UX & Instant Direct Download
# ==============================================================================

import os
import shutil
import subprocess
import glob
import re
import sys
import time
import json
from pathlib import Path

# --- 1. PYTORCH 2.6 COMPATIBILITY PATCH ---
import torch
_orig_torch_load = torch.load
def _safe_torch_load(*args, **kwargs):
    if "weights_only" not in kwargs:
        kwargs["weights_only"] = False
    return _orig_torch_load(*args, **kwargs)
torch.load = _safe_torch_load

# --- 2. ENVIRONMENT INITIALIZATION & MODEL WEIGHTS RECOVERY ---
SETUP_FLAG = "/content/.setup_complete"
WEIGHTS_DIR = "/content/Real-ESRGAN/weights"

MODEL_URLS = {
    "RealESRGAN_x4plus.pth": "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth",
    "RealESRGAN_x2plus.pth": "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth",
    "RealESRGAN_x4plus_anime_6B.pth": "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.2.4/RealESRGAN_x4plus_anime_6B.pth",
    "realesr-animevideov3.pth": "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesr-animevideov3.pth",
    "realesr-general-x4v3.pth": "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesr-general-x4v3.pth",
    "realesr-general-wdn-x4v3.pth": "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesr-general-wdn-x4v3.pth",
    "GFPGANv1.3.pth": "https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.3.pth"
}

def clean_and_verify_weights():
    os.makedirs(WEIGHTS_DIR, exist_ok=True)
    for fname, url in MODEL_URLS.items():
        dest = os.path.join(WEIGHTS_DIR, fname)
        if os.path.exists(dest) and os.path.getsize(dest) < 1_000_000:
            print(f"⚠️ Removing corrupted model file: {fname} ({os.path.getsize(dest)} bytes)", flush=True)
            os.remove(dest)
        
        if not os.path.exists(dest):
            print(f"📥 Downloading verified AI weights: {fname}...", flush=True)
            try:
                torch.hub.download_url_to_file(url, dest, progress=False)
                print(f"✅ Ready: {fname}", flush=True)
            except Exception as e:
                print(f"❌ Failed to download {fname}: {e}", flush=True)

def setup_environment(force=False):
    # 1. Clone repos if missing or incomplete
    if not os.path.exists("/content/Real-ESRGAN/inference_realesrgan_video.py"):
        if os.path.exists("/content/Real-ESRGAN"):
            if os.path.exists("/content/Real-ESRGAN/weights"):
                shutil.copytree("/content/Real-ESRGAN/weights", "/content/temp_weights", dirs_exist_ok=True)
            shutil.rmtree("/content/Real-ESRGAN", ignore_errors=True)
        print("📦 Cloning Real-ESRGAN...", flush=True)
        subprocess.run(["git", "clone", "https://github.com/xinntao/Real-ESRGAN.git", "/content/Real-ESRGAN"], check=True)
        if os.path.exists("/content/temp_weights"):
            os.makedirs("/content/Real-ESRGAN/weights", exist_ok=True)
            shutil.copytree("/content/temp_weights", "/content/Real-ESRGAN/weights", dirs_exist_ok=True)
            shutil.rmtree("/content/temp_weights", ignore_errors=True)

    if not os.path.exists("/content/BasicSR/setup.py"):
        shutil.rmtree("/content/BasicSR", ignore_errors=True)
        print("📦 Cloning BasicSR...", flush=True)
        subprocess.run(["git", "clone", "https://github.com/xinntao/BasicSR.git", "/content/BasicSR"], check=True)

    def patch_file(path, pattern, replacement):
        if os.path.exists(path):
            with open(path, "r", encoding="utf-8") as f: content = f.read()
            content = re.sub(pattern, replacement, content, flags=re.MULTILINE)
            with open(path, "w", encoding="utf-8") as f: f.write(content)

    patch_file("/content/BasicSR/setup.py", r"(\s*)return locals\['__version__'\]", r"\1return '1.4.2'")
    patch_file("/content/BasicSR/setup.py", r"version=get_version\s*(\(.*?\))?", "version='1.4.2'")
    patch_file("/content/Real-ESRGAN/setup.py", r"version=get_version\s*(\(.*?\))?", "version='0.3.0'")

    # Guarantee version.py files exist
    os.makedirs("/content/Real-ESRGAN/realesrgan", exist_ok=True)
    with open("/content/Real-ESRGAN/realesrgan/version.py", "w") as f:
        f.write("__version__ = '0.3.0'\n__gitsha__ = 'default'\n")

    os.makedirs("/content/BasicSR/basicsr", exist_ok=True)
    with open("/content/BasicSR/basicsr/version.py", "w") as f:
        f.write("__version__ = '1.4.2'\n__gitsha__ = 'default'\n")

    # Build development packages
    subprocess.run([sys.executable, "setup.py", "develop"], cwd="/content/BasicSR", check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "ffmpeg-python"], check=False)
    subprocess.run([sys.executable, "setup.py", "develop"], cwd="/content/Real-ESRGAN", check=False)

    patch_targets = (
        glob.glob("/usr/local/lib/python3.*/dist-packages/basicsr/data/degradations.py") +
        glob.glob("/content/BasicSR/basicsr/data/degradations.py")
    )
    for p in patch_targets:
        if os.path.exists(p):
            with open(p, "r", encoding="utf-8") as f: c = f.read().replace("functional_tensor", "functional")
            with open(p, "w", encoding="utf-8") as f: f.write(c)

    # Patch PyTorch 2.6 weights_only unpickler and multiprocessing Pool safety directly into inference script
    inference_script = "/content/Real-ESRGAN/inference_realesrgan_video.py"
    if os.path.exists(inference_script):
        with open(inference_script, "r", encoding="utf-8") as f: inf_code = f.read()
        patch = """import torch
_orig_tl = torch.load
def _safe_load(*args, **kwargs):
    if 'weights_only' not in kwargs: kwargs['weights_only'] = False
    return _orig_tl(*args, **kwargs)
torch.load = _safe_load
if hasattr(torch, 'cuda') and not torch.cuda.is_available():
    torch.cuda.synchronize = lambda *a, **kw: None
"""
                # Clean any old buggy _safe_dev definition
        if "_safe_dev" in inf_code:
            inf_code = re.sub(r'_orig_dev = torch\.device.*?torch\.device = _safe_dev\n?', '', inf_code, flags=re.DOTALL)
        if "_safe_load" not in inf_code:
            inf_code = patch + "\n" + inf_code
        
        # Eliminate ValueError: Number of processes must be at least 1 in Pool
        inf_code = re.sub(r'if num_process == 1:', 'if num_process <= 1:', inf_code)
        inf_code = re.sub(r'num_process\s*=\s*num_gpus\s*\*\s*args\.num_process_per_gpu', 'num_process = max(1, num_gpus * args.num_process_per_gpu)', inf_code)
        
        # Eliminate CUDA synchronization crash on CPU
        inf_code = re.sub(r"device\s*=\s*torch\.device\(['\"]cuda['\"].*?\)", "device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')", inf_code)
        inf_code = inf_code.replace("half=not args.fp32", "half=(not args.fp32 and torch.cuda.is_available())")
        inf_code = re.sub(r"torch\.cuda\.synchronize\(.*?\)", "pass", inf_code)
        
        with open(inference_script, "w", encoding="utf-8") as f: f.write(inf_code)

    clean_and_verify_weights()
    if not os.path.exists("/content/weights") and os.path.exists("/content/Real-ESRGAN/weights"):
        try: os.symlink("/content/Real-ESRGAN/weights", "/content/weights")
        except Exception: pass
    with open(SETUP_FLAG, "w") as f: f.write("ready")
    print("✅ [SYSTEM]: 2026 AI Studio Engine Ready!", flush=True)

# Run environment initialization
setup_environment()

import gradio as gr

# --- 3. METADATA & HELPER FUNCTIONS ---
def extract_video_path(val):
    if not val:
        return None
    if isinstance(val, str):
        return val
    if isinstance(val, dict):
        if "video" in val:
            v = val["video"]
            if isinstance(v, dict):
                return v.get("path")
            elif isinstance(v, str):
                return v
        if "path" in val:
            return val["path"]
    return None

def probe_video(video_path):
    video_path = extract_video_path(video_path)
    if not video_path or not os.path.exists(video_path):
        return None
    try:
        cmd = ["ffprobe", "-v", "error", "-show_entries", 
               "format=duration:stream=width,height,r_frame_rate,codec_name,codec_type,duration,nb_frames", 
               "-of", "json", video_path]
        res = subprocess.run(cmd, capture_output=True, text=True, check=True)
        data = json.loads(res.stdout)
        v = next((s for s in data.get("streams", []) if s.get("codec_type") == "video"), None)
        a = next((s for s in data.get("streams", []) if s.get("codec_type") == "audio"), None)
        w = int(v.get("width", 0)) if v else 0
        h = int(v.get("height", 0)) if v else 0
        dur = float((v.get("duration") if v else None) or data.get("format", {}).get("duration") or 0)
        fps_str = v.get("r_frame_rate", "30/1") if v else "30/1"
        try:
            num, den = map(int, fps_str.split("/"))
            fps = round(num / den, 1)
        except Exception:
            fps = 30.0
        return {"width": w, "height": h, "duration": dur, "fps": fps, "has_audio": a is not None}
    except Exception:
        return None

def format_video_inspector(video_path):
    video_path = extract_video_path(video_path)
    if not video_path:
        return """
        <div class="pro-chip-container empty">
            <span class="pro-chip-icon">ℹ️</span>
            <span>Upload or drag a video to inspect media properties</span>
        </div>
        """
    meta = probe_video(video_path)
    if not meta:
        return """
        <div class="pro-chip-container valid">
            <span class="pro-chip-icon">⚡</span>
            <span>Video loaded successfully</span>
        </div>
        """
    
    sound_chip = '<span class="pro-chip sound-on">🔊 Audio Detected</span>' if meta["has_audio"] else '<span class="pro-chip sound-off">🔇 Muted</span>'
    fps_val = f'{meta["fps"]} FPS'
    dur_str = f'{meta["duration"]:.1f}s' if meta["duration"] > 0 else '< 1s'
    
    return f"""
    <div class="pro-inspector-card">
        <div class="pro-inspector-grid">
            <div class="pro-stat-item">
                <span class="pro-stat-label">ORIGINAL RESOLUTION</span>
                <span class="pro-stat-value">{meta['width']} × {meta['height']}</span>
            </div>
            <div class="pro-stat-item">
                <span class="pro-stat-label">LENGTH</span>
                <span class="pro-stat-value">{dur_str}</span>
            </div>
            <div class="pro-stat-item">
                <span class="pro-stat-label">FRAMERATE</span>
                <span class="pro-stat-value">{fps_val}</span>
            </div>
            <div class="pro-stat-item">
                <span class="pro-stat-label">AUDIO STREAM</span>
                <span class="pro-stat-value">{sound_chip}</span>
            </div>
        </div>
    </div>
    """

# --- 4. HIGH-PERFORMANCE PROCESSING PIPELINE ---
def run_upscale_pipeline(
    video_path,
    preset_choice,
    quality_choice,
    keep_audio,
    quick_test,
    auto_drive,
    advanced_mode,
    pro_model,
    pro_tile,
    pro_codec,
    pro_crf,
    pro_gain,
    progress=gr.Progress()
):
    video_path = extract_video_path(video_path)
    if not video_path or not os.path.exists(video_path):
        return (
            None,
            None,
            gr.update(visible=False),
            """<div class="pro-alert-error">
                <span class="icon">🚨</span>
                <div>
                    <strong>Action Required:</strong> Please upload a source video on the left before clicking Enhance.
                </div>
            </div>"""
        )

    work_dir = "/content/pro_upscale_work"
    out_dir = "/content/pro_upscale_out"
    shutil.rmtree(work_dir, ignore_errors=True)
    shutil.rmtree(out_dir, ignore_errors=True)
    os.makedirs(work_dir, exist_ok=True)
    os.makedirs(out_dir, exist_ok=True)

    import math
    input_file = os.path.join(work_dir, "input.mp4")

    # 1. Video Normalization & 5-Second Preview Slicing
    # Stripping audio (-an) ensures neural input is purely video for maximum pipe stability
    if quick_test:
        progress(0.05, desc="🎬 Creating fast 5s preview sample...")
        subprocess.run(["ffmpeg", "-y", "-i", video_path, "-t", "5", "-c:v", "libx264", "-crf", "14", "-an", input_file], check=True)
    else:
        subprocess.run(["ffmpeg", "-y", "-i", video_path, "-c:v", "libx264", "-crf", "14", "-an", input_file], check=True)

    meta_current = probe_video(input_file)
    total_dur = meta_current["duration"] if meta_current else 0.0

    # 2. Universal Sound Extraction with Gain (Extracted directly from source video)
    temp_audio = os.path.join(work_dir, "audio.m4a")
    has_audio = False
    if keep_audio:
        source_meta = probe_video(video_path)
        if source_meta and source_meta["has_audio"]:
            progress(0.10, desc="🎵 Normalizing audio stream...")
            gain_opts = []
            if advanced_mode and abs(pro_gain - 1.0) > 0.05:
                gain_opts = ["-af", f"volume={pro_gain}"]

            audio_cmd = ["ffmpeg", "-y", "-i", video_path, "-vn"] + gain_opts + ["-c:a", "aac", "-b:a", "256k", temp_audio]
            if quick_test:
                audio_cmd = ["ffmpeg", "-y", "-ss", "0", "-t", "5", "-i", video_path, "-vn"] + gain_opts + ["-c:a", "aac", "-b:a", "256k", temp_audio]
            ret = subprocess.run(audio_cmd, capture_output=True)
            if ret.returncode == 0 and os.path.exists(temp_audio) and os.path.getsize(temp_audio) > 500:
                has_audio = True

    # 3. Model & Neural Scaling Resolution
    if advanced_mode and pro_model:
        model_name = pro_model
        scale = 4 if "x4" in model_name else 2
        tile_size = int(pro_tile)
        codec = "libx264" if "264" in str(pro_codec) else "libx265"
        crf = int(pro_crf)
        face_fix = "Face" in preset_choice or "GFPGAN" in pro_model
    else:
        tile_size = 800
        codec = "libx264"
        crf = 18
        face_fix = False

        if "Anime 6B" in preset_choice or "anime_6B" in str(preset_choice):
            model_name = "RealESRGAN_x4plus_anime_6B"
            scale = 4 if "4K" in quality_choice else 2
            face_fix = False
        elif "Anime" in preset_choice or "animevideov3" in str(preset_choice):
            # Default Anime V3 engine: specifically trained for animated video with zero haloing & sharp lines
            model_name = "realesr-animevideov3"
            scale = 4 if "4K" in quality_choice else 2
            face_fix = False
        elif "Face" in preset_choice:
            model_name = "RealESRGAN_x4plus"
            scale = 4 if "4K" in quality_choice else 2
            face_fix = True
        else:
            # Live Action & Cinema
            model_name = "RealESRGAN_x4plus"
            scale = 4 if "4K" in quality_choice else 2
            face_fix = False

    # 4. Neural Engine Execution (In-Memory Streaming Pipes + Smart Segmenting)
    # Segmenting videos > 180s into 120s chunks keeps VRAM fresh and prevents disk bloat
    segment_sec = 120.0
    use_segments = (total_dur > 180.0) and (not quick_test)

    segments_to_process = []
    if use_segments:
        num_segs = math.ceil(total_dur / segment_sec)
        for s_idx in range(num_segs):
            s_start = s_idx * segment_sec
            s_len = min(segment_sec, total_dur - s_start)
            s_file = os.path.join(work_dir, f"seg_{s_idx:03d}.mp4")
            subprocess.run(["ffmpeg", "-y", "-ss", str(s_start), "-i", input_file, "-t", str(s_len), "-c:v", "libx264", "-crf", "14", "-an", s_file], check=True)
            segments_to_process.append((s_idx, s_file))
    else:
        segments_to_process.append((0, input_file))

    total_chunks = len(segments_to_process)
    upscaled_chunk_files = []

    for chunk_idx, chunk_in in segments_to_process:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        chunk_desc = f"Segment {chunk_idx + 1}/{total_chunks}" if total_chunks > 1 else "Processing"
        progress(0.15 + (chunk_idx / total_chunks) * 0.75, desc=f"🚀 {chunk_desc} on NVIDIA GPU (in-memory pipe)...")

        # In-memory streaming pipe (ZERO PNG frames written to disk)
        cmd = [
            sys.executable, "/content/Real-ESRGAN/inference_realesrgan_video.py",
            "-n", model_name,
            "-i", chunk_in,
            "-o", out_dir,
            "-s", str(scale),
            "--tile", str(tile_size),
            "--num_process_per_gpu", "1"
        ]
        if face_fix:
            cmd.append("--face_enhance")
        if not torch.cuda.is_available():
            cmd.append("--fp32")

        env = dict(os.environ)
        env["TORCH_FORCE_WEIGHTS_ONLY_LOAD"] = "0"
        env["PYTHONPATH"] = f"/content/Real-ESRGAN:/content/BasicSR:{env.get('PYTHONPATH', '')}"

        p = subprocess.Popen(cmd, cwd="/content/Real-ESRGAN", env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        output_lines = []
        for line in p.stdout:
            output_lines.append(line)
            m = re.search(r'(\d+)/(\d+)', line)
            if m:
                curr_f = int(m.group(1))
                tot_f = int(m.group(2))
                if 0 < curr_f <= tot_f and tot_f >= 5:
                    base_p = 0.15 + (chunk_idx / total_chunks) * 0.75
                    span_p = 0.75 / total_chunks
                    pct = int((curr_f / tot_f) * 100)
                    progress(base_p + (curr_f / tot_f) * span_p, desc=f"{chunk_desc}: Frame {curr_f}/{tot_f} ({pct}%)")
        p.wait()

        if p.returncode != 0:
            err_tail = "".join(output_lines[-6:]).replace("<", "&lt;").replace(">", "&gt;")
            return (
                None, None, gr.update(visible=False),
                f"""<div class="pro-alert-error">
                    <span class="icon">🚨</span>
                    <div>
                        <strong>Upscaling Failed ({chunk_desc}):</strong> The neural engine encountered an error.<br>
                        <code style="font-size: 0.75rem; color: #fca5a5;">{err_tail}</code>
                    </div>
                </div>"""
            )

        expected_stem = Path(chunk_in).stem
        candidates = sorted(glob.glob(os.path.join(out_dir, f"{expected_stem}*.mp4")), key=os.path.getmtime)
        if not candidates or os.path.getsize(candidates[-1]) == 0:
            return None, None, gr.update(visible=False), f'<div class="pro-alert-error"><span class="icon">⚠️</span><div>Failed to locate upscaled output for {chunk_desc}.</div></div>'
        upscaled_chunk_files.append(candidates[-1])

    # If chunked, concatenate all chunks losslessly
    if len(upscaled_chunk_files) > 1:
        progress(0.90, desc="⚡ Seamlessly stitching video segments...")
        concat_txt = os.path.join(work_dir, "concat_manifest.txt")
        with open(concat_txt, "w", encoding="utf-8") as f:
            for fpath in upscaled_chunk_files:
                f.write(f"file '{fpath}'\n")
        raw_video = os.path.join(out_dir, "concat_master.mp4")
        subprocess.run(["ffmpeg", "-y", "-f", "concat", "-safe", "0", "-i", concat_txt, "-c", "copy", raw_video], check=True)
    else:
        raw_video = upscaled_chunk_files[0]

    # 5. Master Cinema Packaging with Streaming Faststart
    base_name = Path(video_path).stem
    suffix = "_5s_preview_HD" if quick_test else "_Master_HD"
    final_output = os.path.join(out_dir, f"{base_name}{suffix}.mp4")
    progress(0.92, desc="🎬 Assembling master video container with faststart...")

    ffmpeg_pack = ["ffmpeg", "-y", "-i", raw_video]
    if has_audio:
        ffmpeg_pack.extend(["-i", temp_audio, "-c:a", "copy", "-shortest"])
    else:
        ffmpeg_pack.extend(["-an"])

    ffmpeg_pack.extend([
        "-c:v", codec,
        "-preset", "fast",
        "-crf", str(crf),
        "-pix_fmt", "yuv420p",
        "-movflags", "+faststart",
        final_output
    ])
    subprocess.run(ffmpeg_pack, check=True)

    # 6. Google Drive Sync
    drive_badge = ""
    if auto_drive:
        try:
            drive_path = "/content/drive/MyDrive/AI_Upscaled"
            if os.path.exists("/content/drive/MyDrive"):
                os.makedirs(drive_path, exist_ok=True)
                dest = os.path.join(drive_path, os.path.basename(final_output))
                shutil.copy(final_output, dest)
                drive_badge = '<span class="pro-badge drive-saved">💾 Saved to Google Drive</span>'
        except Exception:
            pass

    meta_out = probe_video(final_output)
    w_out = meta_out['width'] if meta_out else '?'
    h_out = meta_out['height'] if meta_out else '?'
    w_in = meta_current['width'] if meta_current else '?'
    h_in = meta_current['height'] if meta_current else '?'
    sz_mb = os.path.getsize(final_output) / (1024 * 1024)

    success_markup = f"""
    <div class="pro-success-card">
        <div class="pro-success-header">
            <span class="pro-success-icon">✨</span>
            <div class="pro-success-title-box">
                <h4>Enhancement Complete</h4>
                <p>Your master video has been processed and is ready for download.</p>
            </div>
            {drive_badge}
        </div>
        <div class="pro-meta-comparison">
            <div class="comp-col">
                <span class="comp-label">ORIGINAL</span>
                <span class="comp-val">{w_in} × {h_in}</span>
            </div>
            <div class="comp-arrow">➔</div>
            <div class="comp-col highlight">
                <span class="comp-label">ENHANCED MASTER</span>
                <span class="comp-val">{w_out} × {h_out}</span>
            </div>
            <div class="comp-divider"></div>
            <div class="comp-col">
                <span class="comp-label">FILE SIZE</span>
                <span class="comp-val">{sz_mb:.1f} MB</span>
            </div>
        </div>
    </div>
    """

    progress(1.0, desc="🎉 Master Render Ready!")
    return final_output, final_output, gr.update(value=final_output, visible=True), success_markup

# --- 5. UI/UX PRO MAX DESIGN SYSTEM ---
PRO_MAX_THEME = gr.themes.Soft(
    primary_hue="indigo",
    secondary_hue="emerald",
    neutral_hue="slate",
    font=[gr.themes.GoogleFont("Plus Jakarta Sans"), "Inter", "system-ui", "sans-serif"]
)

PRO_MAX_CSS = """
/* ==============================================================================
   UI/UX PRO MAX DESIGN ENGINE (Glassmorphism + Dark Slate + Precision Tokens)
   ============================================================================== */

:root {
    --bg-dark: #07090e;
    --card-surface: rgba(15, 23, 42, 0.72);
    --card-border: rgba(255, 255, 255, 0.08);
    --accent-indigo: #6366f1;
    --accent-emerald: #10b981;
    --accent-cyan: #06b6d4;
    --accent-glow: rgba(99, 102, 241, 0.28);
    --text-primary: #f8fafc;
    --text-secondary: #94a3b8;
    --text-muted: #64748b;
}

body, .gradio-container {
    background-color: var(--bg-dark) !important;
    background-image: 
        radial-gradient(ellipse at 15% 10%, rgba(99, 102, 241, 0.12) 0%, transparent 45%),
        radial-gradient(ellipse at 85% 85%, rgba(16, 185, 129, 0.09) 0%, transparent 50%) !important;
    font-family: 'Plus Jakarta Sans', 'Inter', system-ui, -apple-system, sans-serif !important;
    color: var(--text-primary) !important;
}

/* Glass Card Surfaces */
.pro-card {
    background: var(--card-surface) !important;
    backdrop-filter: blur(18px) !important;
    -webkit-backdrop-filter: blur(18px) !important;
    border: 1px solid var(--card-border) !important;
    border-radius: 16px !important;
    padding: 20px !important;
    box-shadow: 0 10px 32px -8px rgba(0, 0, 0, 0.45) !important;
    transition: all 0.25s cubic-bezier(0.4, 0, 0.2, 1) !important;
    margin-bottom: 16px !important;
}

.pro-card:hover {
    border-color: rgba(99, 102, 241, 0.22) !important;
    box-shadow: 0 12px 36px -8px rgba(99, 102, 241, 0.15) !important;
}

/* Card Header with Badges */
.card-title-row {
    display: flex;
    align-items: center;
    gap: 10px;
    margin-bottom: 14px;
}

.step-pill {
    background: linear-gradient(135deg, #6366f1 0%, #4f46e5 100%);
    color: #ffffff;
    font-size: 0.72rem;
    font-weight: 800;
    letter-spacing: 0.05em;
    padding: 3px 9px;
    border-radius: 20px;
    text-transform: uppercase;
}

.card-heading {
    font-size: 1.05rem;
    font-weight: 700;
    color: var(--text-primary);
    margin: 0;
    letter-spacing: -0.015em;
}

/* Inspector Chips */
.pro-inspector-card {
    background: rgba(10, 15, 29, 0.75);
    border: 1px solid rgba(255, 255, 255, 0.06);
    border-radius: 12px;
    padding: 12px 14px;
    margin-top: 10px;
}

.pro-inspector-grid {
    display: grid;
    grid-template-columns: repeat(2, 1fr);
    gap: 10px;
}

.pro-stat-item {
    display: flex;
    flex-direction: column;
    gap: 2px;
}

.pro-stat-label {
    font-size: 0.68rem;
    font-weight: 700;
    letter-spacing: 0.06em;
    color: var(--text-muted);
}

.pro-stat-value {
    font-size: 0.9rem;
    font-weight: 700;
    color: var(--text-primary);
    font-family: monospace, monospace;
}

.pro-chip-container {
    padding: 10px 14px;
    border-radius: 10px;
    font-size: 0.85rem;
    display: flex;
    align-items: center;
    gap: 8px;
    margin-top: 8px;
}
.pro-chip-container.empty {
    background: rgba(30, 41, 59, 0.4);
    border: 1px dashed rgba(255, 255, 255, 0.1);
    color: var(--text-muted);
}
.pro-chip-container.valid {
    background: rgba(16, 185, 129, 0.12);
    border: 1px solid rgba(16, 185, 129, 0.3);
    color: #34d399;
}

.pro-chip {
    padding: 2px 7px;
    border-radius: 6px;
    font-size: 0.75rem;
    font-weight: 600;
}
.sound-on { background: rgba(16, 185, 129, 0.2); color: #34d399; border: 1px solid rgba(16, 185, 129, 0.3); }
.sound-off { background: rgba(148, 163, 184, 0.15); color: #94a3b8; }

/* Primary Enhancement Action Button */
.pro-cta-btn {
    background: linear-gradient(135deg, #6366f1 0%, #3b82f6 50%, #10b981 100%) !important;
    background-size: 200% auto !important;
    color: #ffffff !important;
    font-size: 1.15rem !important;
    font-weight: 800 !important;
    letter-spacing: -0.01em !important;
    padding: 16px 24px !important;
    border-radius: 14px !important;
    border: none !important;
    box-shadow: 0 8px 26px rgba(99, 102, 241, 0.35) !important;
    width: 100% !important;
    cursor: pointer !important;
    margin-top: 14px !important;
    margin-bottom: 8px !important;
    transition: all 0.3s cubic-bezier(0.4, 0, 0.2, 1) !important;
}

.pro-cta-btn:hover {
    transform: translateY(-2px) scale(1.008) !important;
    box-shadow: 0 12px 34px rgba(99, 102, 241, 0.5) !important;
    background-position: right center !important;
}

.pro-cta-btn:active {
    transform: translateY(1px) scale(0.99) !important;
}

/* Master Download Button */
.pro-download-btn {
    background: linear-gradient(135deg, #10b981 0%, #059669 100%) !important;
    color: #ffffff !important;
    font-size: 1.15rem !important;
    font-weight: 800 !important;
    padding: 16px 24px !important;
    border-radius: 14px !important;
    border: none !important;
    box-shadow: 0 8px 24px rgba(16, 185, 129, 0.4) !important;
    width: 100% !important;
    cursor: pointer !important;
    margin: 12px 0 !important;
    transition: all 0.25s ease !important;
}

.pro-download-btn:hover {
    transform: translateY(-2px) !important;
    box-shadow: 0 12px 30px rgba(16, 185, 129, 0.55) !important;
}

/* Success Card */
.pro-success-card {
    background: rgba(16, 185, 129, 0.09);
    border: 1px solid rgba(16, 185, 129, 0.3);
    border-radius: 14px;
    padding: 16px 20px;
    margin: 12px 0;
}

.pro-success-header {
    display: flex;
    align-items: center;
    gap: 12px;
    margin-bottom: 12px;
}

.pro-success-icon {
    font-size: 1.5rem;
}

.pro-success-title-box h4 {
    margin: 0;
    font-size: 1.1rem;
    font-weight: 800;
    color: #34d399;
}

.pro-success-title-box p {
    margin: 2px 0 0 0;
    font-size: 0.85rem;
    color: var(--text-secondary);
}

.pro-badge.drive-saved {
    margin-left: auto;
    background: rgba(59, 130, 246, 0.2);
    border: 1px solid rgba(59, 130, 246, 0.35);
    color: #60a5fa;
    padding: 3px 10px;
    border-radius: 20px;
    font-size: 0.75rem;
    font-weight: 700;
}

.pro-meta-comparison {
    display: flex;
    align-items: center;
    gap: 16px;
    background: rgba(0, 0, 0, 0.25);
    border-radius: 10px;
    padding: 10px 16px;
}

.comp-col {
    display: flex;
    flex-direction: column;
}

.comp-label {
    font-size: 0.65rem;
    font-weight: 700;
    color: var(--text-muted);
}

.comp-val {
    font-size: 0.95rem;
    font-weight: 800;
    color: var(--text-primary);
}

.comp-col.highlight .comp-val {
    color: #34d399;
}

.comp-arrow {
    color: var(--text-muted);
    font-weight: 900;
}

.comp-divider {
    width: 1px;
    height: 24px;
    background: rgba(255, 255, 255, 0.1);
    margin: 0 8px;
}

/* Error Alert */
.pro-alert-error {
    background: rgba(239, 68, 68, 0.12);
    border: 1px solid rgba(239, 68, 68, 0.35);
    border-radius: 12px;
    padding: 12px 16px;
    color: #fca5a5;
    font-size: 0.9rem;
    display: flex;
    align-items: center;
    gap: 10px;
    margin: 12px 0;
}

/* Placeholder Empty Stage */
.pro-empty-stage {
    display: flex;
    flex-direction: column;
    align-items: center;
    justify-content: center;
    min-height: 280px;
    background: rgba(15, 23, 42, 0.4);
    border: 1px dashed rgba(255, 255, 255, 0.1);
    border-radius: 14px;
    text-align: center;
    padding: 30px;
}

.pro-empty-icon {
    font-size: 2.5rem;
    margin-bottom: 12px;
    filter: drop-shadow(0 4px 10px rgba(99, 102, 241, 0.3));
}

.pro-empty-stage h4 {
    margin: 0 0 6px 0;
    font-size: 1.15rem;
    font-weight: 700;
    color: var(--text-primary);
}

.pro-empty-stage p {
    margin: 0;
    font-size: 0.88rem;
    color: var(--text-muted);
    max-width: 320px;
}
"""

with gr.Blocks(title="OmniUpscaler Pro") as app:
    # Inject CSS via top HTML style block for guaranteed 100% style coverage
    gr.HTML(f"<style>{PRO_MAX_CSS}</style>")

    # Top Brand / Header Showcase
    gr.HTML("""
    <div style="padding: 18px 0 16px 0; border-bottom: 1px solid rgba(255, 255, 255, 0.07); margin-bottom: 20px;">
        <div style="display: flex; align-items: center; justify-content: space-between; flex-wrap: wrap; gap: 12px;">
            <div>
                <div style="display: inline-flex; align-items: center; gap: 8px; background: rgba(99, 102, 241, 0.15); border: 1px solid rgba(99, 102, 241, 0.3); padding: 4px 12px; border-radius: 30px; margin-bottom: 6px;">
                    <span style="display: inline-block; width: 6px; height: 6px; border-radius: 50%; background: #10b981; box-shadow: 0 0 8px #10b981;"></span>
                    <span style="font-size: 0.72rem; font-weight: 800; letter-spacing: 0.08em; color: #a5b4fc; text-transform: uppercase;">Next-Gen Neural Suite</span>
                </div>
                <h1 style="font-size: 2.1rem; font-weight: 900; margin: 0; letter-spacing: -0.03em; background: linear-gradient(90deg, #f8fafc 0%, #cbd5e1 60%, #94a3b8 100%); -webkit-background-clip: text; -webkit-text-fill-color: transparent;">
                    OmniUpscaler Pro
                </h1>
                <p style="margin: 4px 0 0 0; color: #94a3b8; font-size: 0.95rem;">
                    Cinema-grade Super-Resolution, Anime Edge Restoration & High-Fidelity Audio Preservation
                </p>
            </div>
            <div style="display: flex; gap: 8px; flex-wrap: wrap;">
                <div style="background: rgba(30, 41, 59, 0.6); border: 1px solid rgba(255,255,255,0.08); padding: 6px 12px; border-radius: 10px; font-size: 0.78rem; color: #cbd5e1;">
                    ⚡ <strong>NVIDIA T4</strong> GPU
                </div>
                <div style="background: rgba(30, 41, 59, 0.6); border: 1px solid rgba(255,255,255,0.08); padding: 6px 12px; border-radius: 10px; font-size: 0.78rem; color: #cbd5e1;">
                    💎 <strong>Real-ESRGAN v3</strong> + <strong>GFPGAN</strong>
                </div>
            </div>
        </div>
    </div>
    """)

    with gr.Row(equal_height=False):
        # ==============================================================================
        # LEFT COLUMN: CREATIVE CONTROL DECK (Width: 5)
        # ==============================================================================
        with gr.Column(scale=5):
            
            # CARD 1: INPUT VIDEO
            with gr.Group(elem_classes=["pro-card"]):
                gr.HTML("""
                <div class="card-title-row">
                    <span class="step-pill">01</span>
                    <h3 class="card-heading">Select Source Video</h3>
                </div>
                """)
                video_input = gr.Video(label="Drop or Click to Upload Video", sources=["upload"])
                inspector_html = gr.HTML(format_video_inspector(None))
                video_input.change(format_video_inspector, inputs=[video_input], outputs=[inspector_html])

                # HIGH-IMPACT PRIMARY ACTION CTA (FRONT AND CENTER)
                start_btn = gr.Button("✨ Launch AI Upscaler", variant="primary", elem_classes=["pro-cta-btn"])
                quick_preview_toggle = gr.Checkbox(
                    label="⚡ 5-Second Fast Preview Sample (Untick for full video)",
                    value=False,
                    info="Leave unchecked to upscale 100% of your full video. Check only to test a quick 5-second sample."
                )

            # CARD 2: PRESET & PROFILE
            with gr.Group(elem_classes=["pro-card"]):
                gr.HTML("""
                <div class="card-title-row">
                    <span class="step-pill">02</span>
                    <h3 class="card-heading">Enhancement Profile</h3>
                </div>
                """)
                preset_radio = gr.Radio(
                    choices=[
                        "🎨 Anime V3 (realesr-animevideov3 • Razor-Sharp Lines, Zero Ringing)",
                        "✨ Anime 6B (RealESRGAN_x4plus_anime_6B • Deep 2D Illustration Detail)",
                        "🎬 Live Action & Cinema (RealESRGAN_x4plus • Real-World Footage)",
                        "👤 Face Restoration (RealESRGAN_x4plus + GFPGAN • Portraits & Faces)"
                    ],
                    value="🎨 Anime V3 (realesr-animevideov3 • Razor-Sharp Lines, Zero Ringing)",
                    label="AI Enhancement Engine (Default: Anime V3):"
                )

            # CARD 3: QUALITY TARGET & SWITCHES
            with gr.Group(elem_classes=["pro-card"]):
                gr.HTML("""
                <div class="card-title-row">
                    <span class="step-pill">03</span>
                    <h3 class="card-heading">Target Resolution & Speed</h3>
                </div>
                """)
                quality_radio = gr.Radio(
                    choices=[
                        "⚡ 2x HD (Double Resolution) — Recommended (Fast & Crisp)",
                        "🌟 4x Cinema 4K (Maximum Crispness) — High Fidelity"
                    ],
                    value="⚡ 2x HD (Double Resolution) — Recommended (Fast & Crisp)",
                    label="Target Resolution & Speed (Default: 2x Fast HD):"
                )
                
                with gr.Row():
                    audio_toggle = gr.Checkbox(label="🔊 Keep Video Sound", value=True, info="Direct stream preservation")
                    drive_toggle = gr.Checkbox(label="💾 Google Drive Auto-Sync", value=True, info="Saves backup to Drive")

            # CARD 4: COLLAPSIBLE EXPERT DRAWER
            with gr.Accordion("⚙️ Expert Studio Controls & Codec Settings", open=False, elem_classes=["pro-card"]):
                gr.Markdown("*(Optional tuning parameters for power users. Defaults are optimized for all standard clips)*")
                pro_override = gr.Checkbox(label="Enable Manual Parameter Override", value=False)
                pro_model_dropdown = gr.Dropdown(
                    ["RealESRGAN_x4plus", "RealESRGAN_x2plus", "realesr-animevideov3", "RealESRGAN_x4plus_anime_6B", "realesr-general-x4v3"],
                    value="RealESRGAN_x4plus",
                    label="Neural Architecture Checkpoint"
                )
                with gr.Row():
                    pro_tile_slider = gr.Slider(100, 800, value=800, step=50, label="GPU Tile Size (Lower if VRAM runs out)")
                    pro_crf_slider = gr.Slider(14, 28, value=18, step=1, label="CRF Bitrate Quality (Lower = Higher Bitrate)")
                with gr.Row():
                    pro_codec_radio = gr.Radio(["libx264 (H.264 Universal)", "libx265 (HEVC High-Efficiency)"], value="libx264 (H.264 Universal)", label="Export Codec")
                    pro_audio_gain = gr.Slider(0.5, 2.5, value=1.0, step=0.1, label="Audio Gain Normalization")

        # ==============================================================================
        # RIGHT COLUMN: STUDIO DELIVERY & PREVIEW HUB (Width: 7)
        # ==============================================================================
        with gr.Column(scale=7):
            with gr.Group(elem_classes=["pro-card"]):
                gr.HTML("""
                <div class="card-title-row" style="justify-content: space-between;">
                    <div style="display: flex; align-items: center; gap: 10px;">
                        <span class="step-pill" style="background: linear-gradient(135deg, #10b981 0%, #059669 100%);">MASTER</span>
                        <h3 class="card-heading">Enhanced Video Player</h3>
                    </div>
                </div>
                """)

                # RESULT VIDEO PLAYER
                output_player = gr.Video(label="Enhanced Studio Master", interactive=False)

                # UNMISSABLE EMERALD DOWNLOAD BUTTON
                download_master_btn = gr.DownloadButton(
                    "📥 Download Enhanced Master Video (.mp4)",
                    visible=False,
                    elem_classes=["pro-download-btn"]
                )

                # ALTERNATIVE DIRECT FILE CARD
                file_fallback_card = gr.File(label="Alternative Direct File Download", interactive=False)

                # DYNAMIC SUCCESS & COMPARISON CARD
                result_info_html = gr.HTML("""
                <div class="pro-empty-stage">
                    <div class="pro-empty-icon">🎬</div>
                    <h4>Studio Ready</h4>
                    <p>Configure settings on the left and click <strong>Launch AI Upscaler</strong>. Your enhanced master will play and download here.</p>
                </div>
                """)

    # Wire button action
    start_btn.click(
        fn=run_upscale_pipeline,
        inputs=[
            video_input,
            preset_radio,
            quality_radio,
            audio_toggle,
            quick_preview_toggle,
            drive_toggle,
            pro_override,
            pro_model_dropdown,
            pro_tile_slider,
            pro_codec_radio,
            pro_crf_slider,
            pro_audio_gain
        ],
        outputs=[output_player, file_fallback_card, download_master_btn, result_info_html]
    )

app.launch(share=True, inline=True, debug=True, theme=PRO_MAX_THEME)